## 1. Setup: Load Test Data & Engineer Shared Features
Rebuild `DAY_INDEX_NORM` and `RECENT_CASE_COUNT_NORM` on train/val/test exactly as done at training time.

In [9]:
import pandas as pd
import numpy as np
import joblib

# ------------------------------------------------------------
# Step 1: Load train/val/test (needed to compute DAY_INDEX consistently,
# and because RECENT_CASE_COUNT must be computed on train+val+test combined)
# ------------------------------------------------------------
train_df = pd.read_csv("Model 1/artifacts/train_data.csv")
val_df   = pd.read_csv("Model 1/artifacts/val_data.csv")
test_df  = pd.read_csv("Model 1/artifacts/test_data.csv")

for df_ in (train_df, val_df, test_df):
    df_["VISIT_DATE"] = pd.to_datetime(df_["VISIT_DATE"])

# ------------------------------------------------------------
# Step 2: DAY_INDEX_NORM — EXACT same logic as the original notebook
# t0 and day_max must come from train_df ONLY (matches training-time values)
# ------------------------------------------------------------
t0 = train_df["VISIT_DATE"].min()

for df_ in (train_df, val_df, test_df):
    df_["DAY_INDEX"] = (df_["VISIT_DATE"] - t0).dt.days.astype(float)

day_max = train_df["DAY_INDEX"].max()

for df_ in (train_df, val_df, test_df):
    df_["DAY_INDEX_NORM"] = df_["DAY_INDEX"] / day_max

print("t0 (train start date):", t0)
print("day_max (train range in days):", day_max)
print("\nDAY_INDEX_NORM range per split:")
print("Train:", train_df["DAY_INDEX_NORM"].min(), "->", train_df["DAY_INDEX_NORM"].max())
print("Val:  ", val_df["DAY_INDEX_NORM"].min(), "->", val_df["DAY_INDEX_NORM"].max())
print("Test: ", test_df["DAY_INDEX_NORM"].min(), "->", test_df["DAY_INDEX_NORM"].max())

# ------------------------------------------------------------
# Step 3: RECENT_CASE_COUNT — EXACT same causal two-pointer logic
# ------------------------------------------------------------
CASE_RATE_WINDOW_DAYS = 7

def add_recent_case_rate(train_df, val_df, test_df, window_days=CASE_RATE_WINDOW_DAYS):
    train_df = train_df.copy(); train_df["_SPLIT"] = "train"
    val_df   = val_df.copy();   val_df["_SPLIT"]   = "val"
    test_df  = test_df.copy();  test_df["_SPLIT"]  = "test"

    full = pd.concat([train_df, val_df, test_df], ignore_index=True)
    full["VISIT_DATE"] = pd.to_datetime(full["VISIT_DATE"])
    full = full.sort_values("VISIT_DATE").reset_index(drop=True)

    counts = np.zeros(len(full))
    window_np = np.timedelta64(window_days, "D")

    for (city, disease), grp in full.groupby(["CITY", "DISEASE"], sort=False):
        dates = grp["VISIT_DATE"].values
        idx   = grp.index.values
        left  = 0
        for right in range(len(dates)):
            while dates[right] - dates[left] > window_np:
                left += 1
            counts[idx[right]] = right - left  # prior cases in window, excludes current row

    full["RECENT_CASE_COUNT"] = counts

    train_out = full[full["_SPLIT"] == "train"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    val_out   = full[full["_SPLIT"] == "val"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    test_out  = full[full["_SPLIT"] == "test"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    return train_out, val_out, test_out

train_df, val_df, test_df = add_recent_case_rate(train_df, val_df, test_df)

# ------------------------------------------------------------
# Step 4: Apply the SAVED scaler (transform only, NOT fit) — critical
# ------------------------------------------------------------
case_rate_scaler = joblib.load("Model 1/artifacts/case_rate_scaler.pkl")

train_df["RECENT_CASE_COUNT_NORM"] = case_rate_scaler.transform(train_df[["RECENT_CASE_COUNT"]])
val_df["RECENT_CASE_COUNT_NORM"]   = case_rate_scaler.transform(val_df[["RECENT_CASE_COUNT"]])
test_df["RECENT_CASE_COUNT_NORM"]  = case_rate_scaler.transform(test_df[["RECENT_CASE_COUNT"]])

# ------------------------------------------------------------
# Sanity check: compare against the numbers printed in the original notebook
# ------------------------------------------------------------
print("\nSanity check (compare to original notebook output):")
print(f"Mean RECENT_CASE_COUNT — outbreak rows   : "
      f"{train_df.loc[train_df.IS_OUTBREAK==1, 'RECENT_CASE_COUNT'].mean():.2f}")
print(f"Mean RECENT_CASE_COUNT — background rows : "
      f"{train_df.loc[train_df.IS_OUTBREAK==0, 'RECENT_CASE_COUNT'].mean():.2f}")

print("\ntest_df shape:", test_df.shape)
print("New columns present:", [c for c in ["DAY_INDEX_NORM", "RECENT_CASE_COUNT_NORM"] if c in test_df.columns])

t0 (train start date): 2000-01-01 00:16:23
day_max (train range in days): 8289.0

DAY_INDEX_NORM range per split:
Train: 0.0 -> 1.0
Val:   1.0 -> 1.084931837374834
Test:  1.084931837374834 -> 1.1686572566051394


C:\Users\mosta\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Sanity check (compare to original notebook output):
Mean RECENT_CASE_COUNT — outbreak rows   : 291.57
Mean RECENT_CASE_COUNT — background rows : 36.51

test_df shape: (926292, 60)
New columns present: ['DAY_INDEX_NORM', 'RECENT_CASE_COUNT_NORM']


## 2. Task 1–3 Models Applied to the Held-Out Test Set

In [10]:
import pickle

# ------------------------------------------------------------
# Load the trained Isolation Forest and its feature list
# ------------------------------------------------------------
with open("Model 1/artifacts/task3_iso_forest.pkl", "rb") as f:
    iso_forest = pickle.load(f)

SYM_COLS = [c for c in test_df.columns if c.startswith("sym_")]

ISO_FEATURES = SYM_COLS + ["DAY_INDEX_NORM", "AGE_AT_VISIT_NORM",
                            "COMORBIDITY_COUNT_NORM", "RECENT_CASE_COUNT_NORM"]

print("Isolation Forest expects", len(ISO_FEATURES), "features")
print("Model's trained n_features_in_:", iso_forest.n_features_in_)

# Sanity check: feature count must match exactly, or predictions will be silently wrong
assert len(ISO_FEATURES) == iso_forest.n_features_in_, "Feature mismatch!"

# ------------------------------------------------------------
# Run Isolation Forest inference on test_df
# ------------------------------------------------------------
X_iso = test_df[ISO_FEATURES].values

# sklearn's IsolationForest: predict() returns -1 (anomaly) or 1 (normal)
# We convert to 1 = anomaly/outbreak-like, 0 = normal (matches IS_OUTBREAK convention)
raw_pred = iso_forest.predict(X_iso)
test_df["if_pred"] = (raw_pred == -1).astype(int)

# Also keep the continuous anomaly score (lower = more anomalous in sklearn's convention)
test_df["if_score"] = iso_forest.decision_function(X_iso)

print("\nif_pred distribution:")
print(test_df["if_pred"].value_counts())
print(f"\nFlagged rate: {test_df['if_pred'].mean():.4f}")

# Quick check against ground truth
from sklearn.metrics import precision_score, recall_score, f1_score
p = precision_score(test_df["IS_OUTBREAK"], test_df["if_pred"])
r = recall_score(test_df["IS_OUTBREAK"], test_df["if_pred"])
f1 = f1_score(test_df["IS_OUTBREAK"], test_df["if_pred"])
print(f"\nPrecision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print("(compare to metadata: precision=0.1268, recall=0.1189, f1=0.1228)")

C:\Users\mosta\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\mosta\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Isolation Forest expects 32 features
Model's trained n_features_in_: 32

if_pred distribution:
if_pred
0    902460
1     23832
Name: count, dtype: int64

Flagged rate: 0.0257

Precision: 0.1268  Recall: 0.1189  F1: 0.1228
(compare to metadata: precision=0.1268, recall=0.1189, f1=0.1228)


### 2.1 DBSCAN — Apply Saved PCA

In [11]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# Step 1: Load the saved PCA and DBSCAN params
# ------------------------------------------------------------
with open("Model 1/artifacts/task3_pca.pkl", "rb") as f:
    pca = pickle.load(f)

with open("Model 1/artifacts/task3_dbscan_params.pkl", "rb") as f:
    dbscan_params = pickle.load(f)

print("DBSCAN params:", dbscan_params)

# ------------------------------------------------------------
# Step 2: Apply the saved PCA to reduce symptom columns to 8 components
# (transform only, NOT fit — same principle as before)
# ------------------------------------------------------------
pca_cols = [f"pca_{i}" for i in range(8)]
test_df[pca_cols] = pca.transform(test_df[SYM_COLS].values)

DBSCAN_FEATURES_PCA = pca_cols + ["DAY_INDEX_NORM", "RECENT_CASE_COUNT_NORM"]

print("\nDBSCAN features:", DBSCAN_FEATURES_PCA)
print("test_df shape after PCA:", test_df.shape)

C:\Users\mosta\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


DBSCAN params: {'eps': 1.0, 'min_samples': 3}

DBSCAN features: ['pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4', 'pca_5', 'pca_6', 'pca_7', 'DAY_INDEX_NORM', 'RECENT_CASE_COUNT_NORM']
test_df shape after PCA: (926292, 70)


### 2.2 Run Chunked DBSCAN Scoring

In [12]:
import numpy as np

# ------------------------------------------------------------
# Same functions as the original notebook, EXACTLY
# ------------------------------------------------------------
def dbscan_outbreak_score(df_chunk, features, eps, min_samples):
    X = df_chunk[features].values
    if len(X) < min_samples:
        return np.zeros(len(X))

    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)

    scores = np.zeros(len(X))
    unique_labels = set(labels) - {-1}

    for lbl in unique_labels:
        mask = labels == lbl
        cluster_burst = df_chunk.loc[df_chunk.index[mask], "RECENT_CASE_COUNT"].mean()
        scores[mask] = cluster_burst

    return scores


def run_chunked_dbscan(df, features, eps, min_samples):
    scores = np.zeros(len(df))
    df = df.reset_index(drop=True)

    for (city, disease), chunk in df.groupby(["CITY", "DISEASE"]):
        chunk_scores = dbscan_outbreak_score(chunk, features, eps, min_samples)
        scores[chunk.index] = chunk_scores

    return scores

# ------------------------------------------------------------
# Run it on test_df (this will take a while — 64 cities x 18 diseases = up to
# 1152 chunks, each running its own DBSCAN fit)
# ------------------------------------------------------------
print("Running chunked DBSCAN on test_df... this may take several minutes")

test_df = test_df.reset_index(drop=True)
dbscan_scores = run_chunked_dbscan(
    test_df, DBSCAN_FEATURES_PCA,
    eps=dbscan_params["eps"], min_samples=dbscan_params["min_samples"]
)
test_df["dbscan_score"] = dbscan_scores

print("Done.")
print("\ndbscan_score stats:")
print(test_df["dbscan_score"].describe())

Running chunked DBSCAN on test_df... this may take several minutes
Done.

dbscan_score stats:
count    926292.000000
mean         74.344782
std         156.868348
min           0.000000
25%          14.452669
50%          35.527119
75%          69.953024
max        1032.096283
Name: dbscan_score, dtype: float64


### 2.3 Apply DBSCAN Threshold

In [13]:
import json

# ------------------------------------------------------------
# Load the DBSCAN threshold from Task 3's metadata
# ------------------------------------------------------------
with open("Model 1/results/task3_metadata.json", "r") as f:
    task3_metadata = json.load(f)

DBSCAN_THRESHOLD = task3_metadata["models"]["DBSCAN"]["threshold"]
print("DBSCAN_THRESHOLD:", DBSCAN_THRESHOLD)

# ------------------------------------------------------------
# Apply threshold to get binary predictions
# ------------------------------------------------------------
test_df["dbscan_pred"] = (test_df["dbscan_score"] >= DBSCAN_THRESHOLD).astype(int)

print("\ndbscan_pred distribution:")
print(test_df["dbscan_pred"].value_counts())
print(f"\nFlagged rate: {test_df['dbscan_pred'].mean():.4f}")

# ------------------------------------------------------------
# Compare against the original metadata metrics
# ------------------------------------------------------------
p = precision_score(test_df["IS_OUTBREAK"], test_df["dbscan_pred"])
r = recall_score(test_df["IS_OUTBREAK"], test_df["dbscan_pred"])
f1 = f1_score(test_df["IS_OUTBREAK"], test_df["dbscan_pred"])
print(f"\nPrecision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print("(compare to metadata: precision=0.4892, recall=0.4602, f1=0.4742)")

DBSCAN_THRESHOLD: 635.890316038533

dbscan_pred distribution:
dbscan_pred
0    902376
1     23916
Name: count, dtype: int64

Flagged rate: 0.0258

Precision: 0.4892  Recall: 0.4602  F1: 0.4742
(compare to metadata: precision=0.4892, recall=0.4602, f1=0.4742)


### 2.4 Save Combined IF + DBSCAN Predictions

In [14]:
import os

# Make sure the output folder exists
os.makedirs("Task4", exist_ok=True)

# Save test_df with all the new columns (if_pred, dbscan_pred, dbscan_score, etc.)
test_df.to_csv("Task4/test_df_with_if_dbscan_predictions.csv", index=False)

print("Saved: Task4/test_df_with_if_dbscan_predictions.csv")
print("Shape:", test_df.shape)
print("\nColumns added so far:")
print([c for c in test_df.columns if c not in [
    "VISIT_DATE","VISIT_YEAR","VISIT_MONTH","VISIT_WEEK","VISIT_DAY",
    "ENCOUNTERCLASS","GENDER","CITY","STATE","COUNTRY","LAT","LON","SEASON",
    "COMORBIDITY_COUNT","AGE_AT_VISIT","DISEASE","VISIT_NUMBER"
] and not c.startswith("sym_")])

Saved: Task4/test_df_with_if_dbscan_predictions.csv
Shape: (926292, 72)

Columns added so far:
['IS_OUTBREAK', 'HOSPITAL_NAME', 'CITY_ENC', 'COUNTRY_ENC', 'DISEASE_ENC', 'SEASON_ENC', 'GENDER_ENC', 'AGE_AT_VISIT_NORM', 'LAT_NORM', 'LON_NORM', 'COMORBIDITY_COUNT_NORM', 'DAY_INDEX', 'DAY_INDEX_NORM', 'RECENT_CASE_COUNT', 'RECENT_CASE_COUNT_NORM', 'if_pred', 'if_score', 'pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4', 'pca_5', 'pca_6', 'pca_7', 'dbscan_score', 'dbscan_pred']


## 3. Task 2 LSTM Ensemble Applied to the Test Set
### 3.1 Load Models, Threshold, Encoders, Scalers, and Weekly Data

In [17]:
import json
import joblib
from tensorflow import keras

# ------------------------------------------------------------
# Step 1: Load the LSTM threshold
# ------------------------------------------------------------
with open("Model 4/artifacts/best_threshold.json", "r") as f:
    threshold_data = json.load(f)

print("best_threshold.json content:", threshold_data)

# ------------------------------------------------------------
# Step 2: Load the 3 ensemble models (compile=False — inference only)
# ------------------------------------------------------------
model_seed42   = keras.models.load_model("Model 4/checkpoints/best_model_seed42.keras", compile=False)
model_seed123  = keras.models.load_model("Model 4/checkpoints/best_model_seed123.keras", compile=False)
model_seed2024 = keras.models.load_model("Model 4/checkpoints/best_model_seed2024.keras", compile=False)

print("\nAll 3 models loaded successfully")
print("Model input shape:", model_seed42.input_shape)

# ------------------------------------------------------------
# Step 3: Load the encoders and scalers used in Task 2
# ------------------------------------------------------------
lstm_encoders = joblib.load("Model 4/artifacts/encoders.joblib")
lstm_scalers  = joblib.load("Model 4/artifacts/scalers.joblib")

print("\nEncoders keys:", list(lstm_encoders.keys()) if hasattr(lstm_encoders, "keys") else type(lstm_encoders))
print("Scalers keys:", list(lstm_scalers.keys()) if hasattr(lstm_scalers, "keys") else type(lstm_scalers))

# ------------------------------------------------------------
# Step 4: Load the weekly dataset
# ------------------------------------------------------------
weekly_df = pd.read_csv("Model 4/data/weekly_cases_enhanced.csv")
print("\nweekly_df shape:", weekly_df.shape)
print(weekly_df.columns.tolist())

best_threshold.json content: {'best_threshold': 0.28, 'strategy': 'lowest threshold with precision >= 0.6', 'min_precision_floor': 0.6, 'val_precision_at_threshold': 0.6, 'val_recall_at_threshold': 0.7079303675048356, 'val_f1_at_threshold': 0.6495119787045253, 'val_f2_at_threshold': 0.6833457804331591}

All 3 models loaded successfully
Model input shape: [(None, 8, 18), (None, 8), (None, 1), (None, 1), (None, 1), (None, 2)]

Encoders keys: ['country', 'city', 'disease', 'season']
Scalers keys: ['VISIT_YEAR_s', 'LAT_s', 'LON_s', 'case_count_log', 'avg_age_s', 'avg_comorbidity_s', 'cases_2wk_avg_s', 'cases_3wk_avg_s', 'cases_4wk_avg_s', 'cases_4wk_std_s', 'cases_diff_s', 'pct_change_wow_s', 'growth_rate_s', 'case_zscore_s']

weekly_df shape: (983425, 25)
['COUNTRY', 'CITY', 'DISEASE', 'VISIT_WEEK_START', 'VISIT_YEAR', 'VISIT_MONTH', 'SEASON', 'LAT', 'LON', 'case_count', 'outbreak_cases', 'avg_age', 'avg_comorbidity', 'lag_1_cases', 'lag_2_cases', 'lag_3_cases', 'lag_4_cases', 'cases_2wk_

### 3.2 Filter to Diseases Known by the Trained LSTM, Encode & Split

In [20]:

# ------------------------------------------------------------
# Filter to diseases the LSTM encoder actually knows about
# (some low-signal diseases were excluded before training — see Task2 Cell 3)
# ------------------------------------------------------------
known_diseases = set(lstm_encoders["disease"].classes_)
all_diseases_in_data = set(weekly_df["DISEASE"].unique())

excluded = all_diseases_in_data - known_diseases
print("Diseases excluded (unknown to the trained LSTM):", excluded)

rows_before = len(weekly_df)
weekly_df = weekly_df[weekly_df["DISEASE"].isin(known_diseases)].reset_index(drop=True)
print(f"Dropped {rows_before - len(weekly_df)} rows ({(rows_before - len(weekly_df))/rows_before:.1%})")
print("Shape after filtering:", weekly_df.shape)

Diseases excluded (unknown to the trained LSTM): {'Typhoid fever', 'HIV/AIDS', 'Diarrheal disease', 'Tuberculosis', 'Hepatitis A', 'Mpox', 'Leishmaniasis', 'Pneumonia', 'Rift Valley fever'}
Dropped 473763 rows (48.2%)
Shape after filtering: (509662, 27)


In [22]:
# ------------------------------------------------------------
# Apply SAVED encoders (transform only, NOT fit)
# ------------------------------------------------------------
weekly_df["COUNTRY_enc"] = lstm_encoders["country"].transform(weekly_df["COUNTRY"])
weekly_df["CITY_enc"]    = lstm_encoders["city"].transform(weekly_df["CITY"])
weekly_df["DISEASE_enc"] = lstm_encoders["disease"].transform(weekly_df["DISEASE"])
weekly_df["SEASON_enc"]  = lstm_encoders["season"].transform(weekly_df["SEASON"])

# --- Cyclical month encoding ---
weekly_df["month_sin"] = np.sin(2 * np.pi * weekly_df["VISIT_MONTH"] / 12)
weekly_df["month_cos"] = np.cos(2 * np.pi * weekly_df["VISIT_MONTH"] / 12)

# --- Target: next week's outbreak label ---
GROUP_COLS = ["COUNTRY", "CITY", "DISEASE"]
DATE_COL = "VISIT_WEEK_START"
weekly_df[DATE_COL] = pd.to_datetime(weekly_df[DATE_COL])
weekly_df = weekly_df.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)

weekly_df["target_next_week"] = weekly_df.groupby(GROUP_COLS)["label_outbreak_week"].shift(-1)
weekly_df = weekly_df.dropna(subset=["target_next_week"]).reset_index(drop=True)
weekly_df["target_next_week"] = weekly_df["target_next_week"].astype(int)

# --- Recompute the EXACT same 70/15/15 chronological split used in training ---
group_size   = weekly_df.groupby(GROUP_COLS)[DATE_COL].transform("size")
row_in_group = weekly_df.groupby(GROUP_COLS).cumcount()

train_end = (group_size * 0.70).astype(int)
val_end   = (group_size * 0.85).astype(int)

weekly_df["split"] = np.select(
    [row_in_group < train_end, row_in_group < val_end],
    ["train", "val"],
    default="test",
)

print(weekly_df["split"].value_counts())
print("\nTarget positive ratio per split:")
print(weekly_df.groupby("split")["target_next_week"].mean())

split
train    356297
test      76612
val       76345
Name: count, dtype: int64

Target positive ratio per split:
split
test     0.004869
train    0.007247
val      0.006772
Name: target_next_week, dtype: float64


### 3.3 Apply Saved Scalers

In [23]:
# ------------------------------------------------------------
# Apply SAVED scalers (transform only, NOT fit) — replicate Cell 6 logic exactly
# ------------------------------------------------------------

def apply_scale(df, col, out_col, scaler_key, log_transform=False, clip=None):
    work_col = col
    if clip is not None:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan).clip(*clip).fillna(0)
    if log_transform:
        work_col = col + "_log"
        df[work_col] = np.log1p(df[col])
    df[out_col] = lstm_scalers[scaler_key].transform(df[[work_col]])

# static numerics
apply_scale(weekly_df, "VISIT_YEAR", "VISIT_YEAR_s", "VISIT_YEAR_s")
apply_scale(weekly_df, "LAT", "LAT_s", "LAT_s")
apply_scale(weekly_df, "LON", "LON_s", "LON_s")

# case_count family -> shared scaler "case_count_log"
apply_scale(weekly_df, "case_count", "case_count_s", "case_count_log", log_transform=True)

for lag_col in ["lag_1_cases", "lag_2_cases", "lag_3_cases", "lag_4_cases"]:
    weekly_df[lag_col + "_log"] = np.log1p(weekly_df[lag_col])
    weekly_df[lag_col + "_s"] = lstm_scalers["case_count_log"].transform(
        weekly_df[[lag_col + "_log"]].rename(columns={lag_col + "_log": "case_count_log"})
    )

apply_scale(weekly_df, "avg_age", "avg_age_s", "avg_age_s")
apply_scale(weekly_df, "avg_comorbidity", "avg_comorbidity_s", "avg_comorbidity_s")

for col in ["cases_2wk_avg", "cases_3wk_avg", "cases_4wk_avg", "cases_4wk_std"]:
    apply_scale(weekly_df, col, col + "_s", col + "_s", log_transform=True)

apply_scale(weekly_df, "cases_diff", "cases_diff_s", "cases_diff_s", clip=(-100, 100))
apply_scale(weekly_df, "pct_change_wow", "pct_change_wow_s", "pct_change_wow_s", clip=(-5, 5))
apply_scale(weekly_df, "growth_rate", "growth_rate_s", "growth_rate_s", clip=(-5, 5))

# CUSUM/EWMA-style z-score (computed from raw lag columns, then scaled)
lag_cols = ["lag_1_cases", "lag_2_cases", "lag_3_cases", "lag_4_cases"]
lag_mean = weekly_df[lag_cols].mean(axis=1)
lag_std  = weekly_df[lag_cols].std(axis=1)
weekly_df["case_zscore"] = (weekly_df["case_count"] - lag_mean) / (lag_std + 1)
apply_scale(weekly_df, "case_zscore", "case_zscore_s", "case_zscore_s", clip=(-30, 30))

print("Scaling complete. New columns sample:")
print(weekly_df[["case_count_s", "avg_age_s", "cases_diff_s", "case_zscore_s"]].describe())

Scaling complete. New columns sample:
        case_count_s      avg_age_s   cases_diff_s  case_zscore_s
count  509254.000000  509254.000000  509254.000000  509254.000000
mean        0.338041       0.079326      -0.000767       0.009832
std         1.169013       0.928130       1.395130       1.279655
min        -1.011599      -1.166570     -19.335098     -25.393007
25%        -1.011599      -1.166570      -0.193404      -0.461611
50%         0.048177       0.397054      -0.000054      -0.069860
75%         1.107953       0.775569       0.193297       0.352192
max         6.489629       2.430441      19.334990      25.253286


### 3.4 Build Sequence Windows (SEQ_LEN=8)

In [25]:
# ------------------------------------------------------------
# Windowing — build sequences of length SEQ_LEN=8 per (COUNTRY, CITY, DISEASE) series
# Only need windows whose TARGET row falls in the test split
# ------------------------------------------------------------
SEQ_LEN = 8

SEQ_NUMERIC_COLS = [
    "VISIT_YEAR_s", "month_sin", "month_cos",
    "case_count_s", "avg_age_s", "avg_comorbidity_s",
    "lag_1_cases_s", "lag_2_cases_s", "lag_3_cases_s", "lag_4_cases_s",
    "cases_2wk_avg_s", "cases_3wk_avg_s", "cases_4wk_avg_s", "cases_4wk_std_s",
    "cases_diff_s", "pct_change_wow_s", "growth_rate_s", "case_zscore_s"
]

def build_windows_for_group(g):
    n = len(g)
    if n < SEQ_LEN:
        return None

    seq_numeric = g[SEQ_NUMERIC_COLS].values.astype(np.float32)
    season_seq  = g["SEASON_enc"].values.astype(np.int32)
    target      = g["target_next_week"].values.astype(np.int8)
    split_col   = g["split"].values
    dates       = g["VISIT_WEEK_START"].values

    country_id = g["COUNTRY_enc"].iloc[0]
    city_id    = g["CITY_enc"].iloc[0]
    disease_id = g["DISEASE_enc"].iloc[0]
    lat_s      = g["LAT_s"].iloc[0]
    lon_s      = g["LON_s"].iloc[0]

    n_windows = n - SEQ_LEN + 1
    out = {"seq_numeric": [], "season_seq": [], "country": [], "city": [], "disease": [],
           "lat_lon": [], "y": [], "split": [], "target_date": [], "group": []}
    group_key = f"{g['COUNTRY'].iloc[0]}|{g['CITY'].iloc[0]}|{g['DISEASE'].iloc[0]}"

    for i in range(n_windows):
        end = i + SEQ_LEN - 1
        out["seq_numeric"].append(seq_numeric[i:i+SEQ_LEN])
        out["season_seq"].append(season_seq[i:i+SEQ_LEN])
        out["country"].append(country_id)
        out["city"].append(city_id)
        out["disease"].append(disease_id)
        out["lat_lon"].append([lat_s, lon_s])
        out["y"].append(target[end])
        out["split"].append(split_col[end])
        out["target_date"].append(dates[end])
        out["group"].append(group_key)
    return out

all_data = {k: [] for k in ["seq_numeric", "season_seq", "country", "city",
                             "disease", "lat_lon", "y", "split", "target_date", "group"]}

for keys, g in weekly_df.groupby(GROUP_COLS):
    res = build_windows_for_group(g)
    if res is None:
        continue
    for k in all_data:
        all_data[k].extend(res[k])

X_seq_numeric = np.array(all_data["seq_numeric"], dtype=np.float32)
X_season      = np.array(all_data["season_seq"], dtype=np.int32)
X_country     = np.array(all_data["country"], dtype=np.int32)
X_city        = np.array(all_data["city"], dtype=np.int32)
X_disease     = np.array(all_data["disease"], dtype=np.int32)
X_latlon      = np.array(all_data["lat_lon"], dtype=np.float32)
y_all         = np.array(all_data["y"], dtype=np.int8)
split_all     = np.array(all_data["split"])
target_dates  = np.array(all_data["target_date"])
group_all     = np.array(all_data["group"])

print("Total windows:", len(y_all))
print("Split counts:", pd.Series(split_all).value_counts().to_dict())
print("Sequence shape:", X_seq_numeric.shape)

Total windows: 506398
Split counts: {'train': 353441, 'test': 76612, 'val': 76345}
Sequence shape: (506398, 8, 18)


### 3.5 Ensemble Inference & Evaluation vs Training-Time Metrics

In [26]:
# ------------------------------------------------------------
# Build test-only input dict and run ensemble inference
# ------------------------------------------------------------
test_mask = split_all == "test"

test_data = {
    "seq_numeric": X_seq_numeric[test_mask],
    "season":      X_season[test_mask],
    "country":     X_country[test_mask],
    "city":        X_city[test_mask],
    "disease":     X_disease[test_mask],
    "latlon":      X_latlon[test_mask],
    "y":           y_all[test_mask],
}

def to_input_dict(d):
    return {"seq_numeric": d["seq_numeric"], "season_seq": d["season"],
            "country": d["country"], "city": d["city"], "disease": d["disease"], "latlon": d["latlon"]}

ensemble_models = [model_seed42, model_seed123, model_seed2024]

test_probs_per_model = [m.predict(to_input_dict(test_data), verbose=0).ravel() for m in ensemble_models]
lstm_prob = np.mean(test_probs_per_model, axis=0)
y_test = test_data["y"]

LSTM_THRESHOLD = threshold_data["best_threshold"]  # 0.28
lstm_pred = (lstm_prob >= LSTM_THRESHOLD).astype(int)

# ------------------------------------------------------------
# Sanity check against final_metrics.json
# ------------------------------------------------------------
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

p  = precision_score(y_test, lstm_pred)
r  = recall_score(y_test, lstm_pred)
f1 = f1_score(y_test, lstm_pred)
auc = roc_auc_score(y_test, lstm_prob)

print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")
print("(compare to final_metrics.json: precision=0.7667, recall=0.5550, f1=0.6439, roc_auc=0.8847)")

from sklearn.metrics import confusion_matrix
print("\nConfusion matrix:")
print(confusion_matrix(y_test, lstm_pred))
print("(compare to original: [[76176, 63], [166, 207]])")

Precision: 0.7667  Recall: 0.5550  F1: 0.6439  ROC-AUC: 0.8847
(compare to final_metrics.json: precision=0.7667, recall=0.5550, f1=0.6439, roc_auc=0.8847)

Confusion matrix:
[[76176    63]
 [  166   207]]
(compare to original: [[76176, 63], [166, 207]])


### 3.6 Save LSTM Test Predictions

In [27]:
import numpy as np

# ------------------------------------------------------------
# Save LSTM test results as a checkpoint (weekly-level, test split only)
# ------------------------------------------------------------
lstm_results_df = pd.DataFrame({
    "group":       group_all[test_mask],
    "target_date": target_dates[test_mask],
    "y_true":      y_test,
    "lstm_prob":   lstm_prob,
    "lstm_pred":   lstm_pred,
})

# Split "group" (COUNTRY|CITY|DISEASE) back into separate columns for easier merging later
lstm_results_df[["COUNTRY", "CITY", "DISEASE"]] = lstm_results_df["group"].str.split("|", expand=True)

lstm_results_df.to_csv("Task4/lstm_test_predictions.csv", index=False)

print("Saved: Task4/lstm_test_predictions.csv")
print("Shape:", lstm_results_df.shape)
print(lstm_results_df.head())

Saved: Task4/lstm_test_predictions.csv
Shape: (76612, 8)
                      group target_date  y_true  lstm_prob  lstm_pred  \
0  Algeria|Algiers|COVID-19  2025-05-12       0   0.073565          0   
1  Algeria|Algiers|COVID-19  2025-05-19       0   0.071207          0   
2  Algeria|Algiers|COVID-19  2025-05-26       0   0.067156          0   
3  Algeria|Algiers|COVID-19  2025-06-02       0   0.059610          0   
4  Algeria|Algiers|COVID-19  2025-06-09       0   0.057684          0   

   COUNTRY     CITY   DISEASE  
0  Algeria  Algiers  COVID-19  
1  Algeria  Algiers  COVID-19  
2  Algeria  Algiers  COVID-19  
3  Algeria  Algiers  COVID-19  
4  Algeria  Algiers  COVID-19  


## 4. Fuse IF + DBSCAN + LSTM into Weekly Alerts
### 4.1 Aggregate IF/DBSCAN to Weekly Level

In [28]:
import pandas as pd

# ------------------------------------------------------------
# Step 1: Convert each visit's date to its ISO week start (Monday)
# to match the weekly LSTM date format
# ------------------------------------------------------------
test_df["VISIT_DATE"] = pd.to_datetime(test_df["VISIT_DATE"])
test_df["WEEK_START"] = test_df["VISIT_DATE"] - pd.to_timedelta(test_df["VISIT_DATE"].dt.weekday, unit="D")
test_df["WEEK_START"] = test_df["WEEK_START"].dt.normalize()  # strip time component

# Quick sanity check: does this match the LSTM date format style?
print("Sample WEEK_START values:")
print(test_df[["VISIT_DATE", "WEEK_START"]].head())
print("\nIs 2025-05-12 a Monday? ->", pd.Timestamp("2025-05-12").day_name())

# ------------------------------------------------------------
# Step 2: Aggregate IF and DBSCAN to (CITY, DISEASE, WEEK_START) level
# Rule: a week is "flagged" if AT LEAST ONE visit that week was flagged
# ------------------------------------------------------------
weekly_agg = test_df.groupby(["CITY", "DISEASE", "WEEK_START"]).agg(
    if_flagged_visits=("if_pred", "sum"),
    dbscan_flagged_visits=("dbscan_pred", "sum"),
    total_visits=("if_pred", "count"),
    true_outbreak=("IS_OUTBREAK", "max"),
).reset_index()

weekly_agg["IF_FLAG"] = (weekly_agg["if_flagged_visits"] >= 1).astype(int)
weekly_agg["DBSCAN_FLAG"] = (weekly_agg["dbscan_flagged_visits"] >= 1).astype(int)

print("\nweekly_agg shape:", weekly_agg.shape)
print(weekly_agg.head())

Sample WEEK_START values:
           VISIT_DATE WEEK_START
0 2024-08-15 18:06:28 2024-08-12
1 2024-08-15 18:14:44 2024-08-12
2 2024-08-15 18:17:03 2024-08-12
3 2024-08-15 18:17:14 2024-08-12
4 2024-08-15 18:18:26 2024-08-12

Is 2025-05-12 a Monday? -> Monday

weekly_agg shape: (70235, 9)
    CITY   DISEASE WEEK_START  if_flagged_visits  dbscan_flagged_visits  \
0  Abuja  COVID-19 2024-08-12                  0                      0   
1  Abuja  COVID-19 2024-08-19                  0                      0   
2  Abuja  COVID-19 2024-08-26                  0                      0   
3  Abuja  COVID-19 2024-09-02                  0                      0   
4  Abuja  COVID-19 2024-09-09                  1                      0   

   total_visits  true_outbreak  IF_FLAG  DBSCAN_FLAG  
0             4              0        0            0  
1             6              0        0            0  
2             7              0        0            0  
3             1              0        0 

### 4.2 Merge with LSTM Weekly Predictions

In [29]:
# ------------------------------------------------------------
# Merge LSTM (weekly, test-only) with IF/DBSCAN weekly aggregation
# ------------------------------------------------------------
lstm_results_df["target_date"] = pd.to_datetime(lstm_results_df["target_date"])

merged = pd.merge(
    lstm_results_df[["CITY", "DISEASE", "target_date", "y_true", "lstm_prob", "lstm_pred"]],
    weekly_agg[["CITY", "DISEASE", "WEEK_START", "IF_FLAG", "DBSCAN_FLAG", "true_outbreak", "total_visits"]],
    left_on=["CITY", "DISEASE", "target_date"],
    right_on=["CITY", "DISEASE", "WEEK_START"],
    how="left"  # keep all LSTM rows; some weeks might have zero visits in test_df
)

print("Merged shape:", merged.shape)
print("Rows with no matching IF/DBSCAN data (no visits that week):", merged["IF_FLAG"].isna().sum())

# Weeks with no visits at all -> treat as "no anomaly" (0), not missing
merged["IF_FLAG"] = merged["IF_FLAG"].fillna(0).astype(int)
merged["DBSCAN_FLAG"] = merged["DBSCAN_FLAG"].fillna(0).astype(int)

print("\nFlag distributions after merge:")
print("IF_FLAG:    ", merged["IF_FLAG"].value_counts().to_dict())
print("DBSCAN_FLAG:", merged["DBSCAN_FLAG"].value_counts().to_dict())
print("lstm_pred:  ", merged["lstm_pred"].value_counts().to_dict())

merged.to_csv("Task4/merged_weekly_signals.csv", index=False)
print("\nSaved: Task4/merged_weekly_signals.csv")

Merged shape: (76612, 11)
Rows with no matching IF/DBSCAN data (no visits that week): 40896

Flag distributions after merge:
IF_FLAG:     {0: 67740, 1: 8872}
DBSCAN_FLAG: {0: 76455, 1: 157}
lstm_pred:   {0: 76342, 1: 270}

Saved: Task4/merged_weekly_signals.csv


### 4.3 Restrict to the Overlapping Test Period

In [31]:
# ------------------------------------------------------------
# Confirm: what fraction of LSTM test rows fall BEFORE the global IF/DBSCAN cutoff?
# ------------------------------------------------------------
GLOBAL_CUTOFF = pd.Timestamp("2024-08-15")

before_cutoff = (lstm_results_df["target_date"] < GLOBAL_CUTOFF).sum()
after_cutoff  = (lstm_results_df["target_date"] >= GLOBAL_CUTOFF).sum()

print(f"LSTM test rows BEFORE global cutoff (2024-08-15): {before_cutoff} ({before_cutoff/len(lstm_results_df):.1%})")
print(f"LSTM test rows AFTER/ON global cutoff:            {after_cutoff} ({after_cutoff/len(lstm_results_df):.1%})")

print(f"\nCompare to unmatched rows we found: 40,896 ({40896/76612:.1%})")

LSTM test rows BEFORE global cutoff (2024-08-15): 39135 (51.1%)
LSTM test rows AFTER/ON global cutoff:            37477 (48.9%)

Compare to unmatched rows we found: 40,896 (53.4%)


In [32]:
# ------------------------------------------------------------
# Restrict the merged dataset to the overlapping period only
# (the only window where all 3 models have valid predictions)
# ------------------------------------------------------------
merged_overlap = merged[merged["target_date"] >= GLOBAL_CUTOFF].copy()

print("Rows before restricting:", len(merged), "-> after:", len(merged_overlap))
print("\nRemaining rows still missing IF/DBSCAN data (should be small now):")
print(merged_overlap["IF_FLAG"].isna().sum() if merged_overlap["IF_FLAG"].isna().any() else "0 (already filled earlier)")

print("\nFlag distributions in the overlap period:")
print("IF_FLAG:    ", merged_overlap["IF_FLAG"].value_counts().to_dict())
print("DBSCAN_FLAG:", merged_overlap["DBSCAN_FLAG"].value_counts().to_dict())
print("lstm_pred:  ", merged_overlap["lstm_pred"].value_counts().to_dict())
print("\ntrue_outbreak (from IF/DBSCAN side) vs y_true (from LSTM side) — should mostly agree:")
print("Correlation check - rows where they disagree:",
      (merged_overlap["true_outbreak"].fillna(-1) != merged_overlap["y_true"]).sum())

merged_overlap.to_csv("Task4/merged_weekly_signals_overlap.csv", index=False)
print("\nSaved: Task4/merged_weekly_signals_overlap.csv")

Rows before restricting: 76612 -> after: 37477

Remaining rows still missing IF/DBSCAN data (should be small now):
0 (already filled earlier)

Flag distributions in the overlap period:
IF_FLAG:     {0: 28651, 1: 8826}
DBSCAN_FLAG: {0: 37321, 1: 156}
lstm_pred:   {0: 37401, 1: 76}

true_outbreak (from IF/DBSCAN side) vs y_true (from LSTM side) — should mostly agree:
Correlation check - rows where they disagree: 2099

Saved: Task4/merged_weekly_signals_overlap.csv


### 4.4 Finalize the Ground-Truth Column

In [33]:
# ------------------------------------------------------------
# Standardize on true_outbreak as the single ground-truth column
# (y_true was LSTM's forward-looking training target — different semantics)
# ------------------------------------------------------------
merged_overlap = merged_overlap.rename(columns={"y_true": "lstm_target_next_week"})
merged_overlap["true_outbreak"] = merged_overlap["true_outbreak"].fillna(0).astype(int)

print("Final columns:", merged_overlap.columns.tolist())
print("\nGround truth (true_outbreak) distribution in overlap period:")
print(merged_overlap["true_outbreak"].value_counts())

merged_overlap.to_csv("Task4/merged_weekly_signals_overlap.csv", index=False)
print("\nSaved (updated).")

Final columns: ['CITY', 'DISEASE', 'target_date', 'lstm_target_next_week', 'lstm_prob', 'lstm_pred', 'WEEK_START', 'IF_FLAG', 'DBSCAN_FLAG', 'true_outbreak', 'total_visits']

Ground truth (true_outbreak) distribution in overlap period:
true_outbreak
0    37354
1      123
Name: count, dtype: int64

Saved (updated).


### 4.5 Apply Weighted Fusion & Assign Alert Levels

In [34]:
# ------------------------------------------------------------
# Apply the Weighted Fusion scoring system
# ------------------------------------------------------------
merged_overlap["fusion_score"] = (
    merged_overlap["IF_FLAG"] * 1 +
    merged_overlap["DBSCAN_FLAG"] * 2 +
    merged_overlap["lstm_pred"] * 2
)

def assign_alert(score):
    if score == 0:
        return "Normal"
    elif score <= 2:
        return "Watch"
    elif score == 3:
        return "Warning"
    else:  # 4 or 5
        return "Emergency"

merged_overlap["alert_level"] = merged_overlap["fusion_score"].apply(assign_alert)

print("Alert level distribution:")
print(merged_overlap["alert_level"].value_counts())

print("\nAlert level vs actual outbreak (cross-tab):")
print(pd.crosstab(merged_overlap["alert_level"], merged_overlap["true_outbreak"]))

# ------------------------------------------------------------
# Quick sanity check: precision at each alert level
# (what fraction of alerts at this level were TRUE outbreaks?)
# ------------------------------------------------------------
print("\nPrecision per alert level (of rows AT that level, how many were real outbreaks):")
for level in ["Watch", "Warning", "Emergency"]:
    subset = merged_overlap[merged_overlap["alert_level"] == level]
    if len(subset) > 0:
        precision = subset["true_outbreak"].mean()
        print(f"  {level}: {precision:.2%} ({subset['true_outbreak'].sum()}/{len(subset)})")

merged_overlap.to_csv("Task4/task4_weekly_alerts.csv", index=False)
print("\nSaved: Task4/task4_weekly_alerts.csv")

Alert level distribution:
alert_level
Normal       28563
Watch         8780
Warning        124
Emergency       10
Name: count, dtype: int64

Alert level vs actual outbreak (cross-tab):
true_outbreak      0   1
alert_level             
Emergency          1   9
Normal         28519  44
Warning           76  48
Watch           8758  22

Precision per alert level (of rows AT that level, how many were real outbreaks):
  Watch: 0.25% (22/8780)
  Emergency: 90.00% (9/10)

Saved: Task4/task4_weekly_alerts.csv


## 5. Validate Against 12 Historical Outbreak Scenarios
### 5.1 Identify Exact Outbreak Dates per Scenario

In [35]:
import pandas as pd

# ------------------------------------------------------------
# Step 1: Load train + val + test combined (need full 2000-2026 coverage,
# since some historical scenarios fall in the train period)
# ------------------------------------------------------------
train_df_full = pd.read_csv("Model 1/artifacts/train_data.csv", usecols=["CITY", "DISEASE", "VISIT_DATE", "IS_OUTBREAK"])
val_df_full   = pd.read_csv("Model 1/artifacts/val_data.csv",   usecols=["CITY", "DISEASE", "VISIT_DATE", "IS_OUTBREAK"])
# test_df already in memory, just select the needed columns
test_subset   = test_df[["CITY", "DISEASE", "VISIT_DATE", "IS_OUTBREAK"]].copy()

full_outbreak_data = pd.concat([train_df_full, val_df_full, test_subset], ignore_index=True)
full_outbreak_data["VISIT_DATE"] = pd.to_datetime(full_outbreak_data["VISIT_DATE"])

print("Combined shape:", full_outbreak_data.shape)

# ------------------------------------------------------------
# Step 2: The 12 known historical scenarios (city, disease, approximate period)
# ------------------------------------------------------------
scenarios = [
    ("Kinshasa", "Ebola virus disease", "2020-08-01", "2020-10-31"),
    ("Lagos", "Lassa fever", "2021-01-01", "2021-03-31"),
    ("Nairobi", "Cholera", "2021-04-01", "2021-06-30"),
    ("Addis Ababa", "Malaria", "2021-09-01", "2021-11-30"),
    ("Kampala", "Ebola virus disease", "2022-09-01", "2022-11-30"),
    ("Khartoum", "Cholera", "2022-07-01", "2022-09-30"),
    ("Cairo", "COVID-19", "2021-01-01", "2021-04-30"),
    ("Goma", "Mpox", "2023-01-01", "2023-04-30"),
    ("Dar es Salaam", "Dengue fever", "2022-03-01", "2022-06-30"),
    ("Kano", "Meningitis", "2021-12-01", "2022-02-28"),
    ("Mansoura", "Typhoid fever", "2022-05-01", "2022-07-31"),
    ("Lusaka", "Cholera", "2023-01-01", "2023-03-31"),
]

# ------------------------------------------------------------
# Step 3: For each scenario, find the exact outbreak run that overlaps
# the documented approximate window
# ------------------------------------------------------------
def find_exact_outbreak_run(df, city, disease, approx_start, approx_end):
    subset = df[(df["CITY"] == city) & (df["DISEASE"] == disease) & (df["IS_OUTBREAK"] == 1)].copy()
    if len(subset) == 0:
        return None

    subset = subset.sort_values("VISIT_DATE").reset_index(drop=True)
    # Split into separate "runs" wherever the gap between consecutive outbreak
    # visits exceeds 14 days
    subset["gap_days"] = subset["VISIT_DATE"].diff().dt.days.fillna(0)
    subset["run_id"] = (subset["gap_days"] > 14).cumsum()

    runs = subset.groupby("run_id")["VISIT_DATE"].agg(["min", "max", "count"]).reset_index()

    approx_start, approx_end = pd.Timestamp(approx_start), pd.Timestamp(approx_end)
    # Pick the run that overlaps the documented approximate window
    overlapping = runs[(runs["min"] <= approx_end) & (runs["max"] >= approx_start)]

    if len(overlapping) == 0:
        return None
    # If multiple overlap (rare), pick the one with the most records
    best_run = overlapping.sort_values("count", ascending=False).iloc[0]
    return best_run["min"], best_run["max"], int(best_run["count"])

results = []
for city, disease, approx_start, approx_end in scenarios:
    result = find_exact_outbreak_run(full_outbreak_data, city, disease, approx_start, approx_end)
    if result is None:
        results.append({"CITY": city, "DISEASE": disease, "exact_start": None, "exact_end": None,
                         "n_records": 0, "status": "NOT FOUND"})
    else:
        exact_start, exact_end, n_records = result
        results.append({"CITY": city, "DISEASE": disease, "exact_start": exact_start, "exact_end": exact_end,
                         "n_records": n_records, "status": "OK"})

scenarios_df = pd.DataFrame(results)
print(scenarios_df)

scenarios_df.to_csv("Task4/scenarios_exact_dates.csv", index=False)
print("\nSaved: Task4/scenarios_exact_dates.csv")

Combined shape: (6175277, 4)
             CITY              DISEASE exact_start  exact_end  n_records  \
0        Kinshasa  Ebola virus disease  2020-08-02 2020-10-29       3000   
1           Lagos          Lassa fever  2021-01-02 2021-03-30       2500   
2         Nairobi              Cholera  2021-04-01 2021-06-29       4000   
3     Addis Ababa              Malaria  2021-09-01 2021-11-29       5000   
4         Kampala  Ebola virus disease  2022-09-03 2022-11-29       2000   
5        Khartoum              Cholera  2022-07-01 2022-09-29       3500   
6           Cairo             COVID-19  2021-01-02 2021-04-29       6000   
7            Goma                 Mpox  2023-01-03 2023-04-29       2500   
8   Dar es Salaam         Dengue fever  2022-03-04 2022-06-29       3000   
9            Kano           Meningitis  2021-12-01 2022-02-27       2000   
10       Mansoura        Typhoid fever  2022-05-03 2022-07-30       1500   
11         Lusaka              Cholera  2023-01-02 2023-03-

### 5.2 Initial Validation (Global Test-Period Signals)

In [36]:
# ------------------------------------------------------------
# Validate each of the 12 scenarios against the weekly alert system
# ------------------------------------------------------------
alerts_df = pd.read_csv("Task4/task4_weekly_alerts.csv")
alerts_df["target_date"] = pd.to_datetime(alerts_df["target_date"])

LSTM_EXCLUDED_DISEASES = {"Mpox", "Typhoid fever", "HIV/AIDS", "Diarrheal disease",
                           "Tuberculosis", "Hepatitis A", "Leishmaniasis",
                           "Pneumonia", "Rift Valley fever"}

alert_rank = {"Normal": 0, "Watch": 1, "Warning": 2, "Emergency": 3}

validation_results = []

for _, row in scenarios_df.iterrows():
    city, disease = row["CITY"], row["DISEASE"]
    exact_start = row["exact_start"]

    city_disease_alerts = alerts_df[(alerts_df["CITY"] == city) & (alerts_df["DISEASE"] == disease)].sort_values("target_date")

    if len(city_disease_alerts) == 0:
        validation_results.append({
            "CITY": city, "DISEASE": disease, "in_test_period": False,
            "lstm_excluded": disease in LSTM_EXCLUDED_DISEASES,
            "first_alert_date": None, "lead_time_weeks": None,
            "max_alert_level": None, "detected": False,
        })
        continue

    # Look for the first alert (any level above Normal) within 8 weeks BEFORE exact_start
    lookback_window = city_disease_alerts[
        (city_disease_alerts["target_date"] >= exact_start - pd.Timedelta(weeks=8)) &
        (city_disease_alerts["target_date"] < exact_start)
    ]
    warning_rows = lookback_window[lookback_window["alert_level"] != "Normal"]

    if len(warning_rows) > 0:
        first_alert_date = warning_rows["target_date"].min()
        lead_time_weeks = (exact_start - first_alert_date).days / 7
    else:
        first_alert_date, lead_time_weeks = None, 0

    # Highest alert level reached during the outbreak period itself
    during_outbreak = city_disease_alerts[
        (city_disease_alerts["target_date"] >= exact_start) &
        (city_disease_alerts["target_date"] <= row["exact_end"])
    ]
    if len(during_outbreak) > 0:
        max_level = during_outbreak["alert_level"].map(alert_rank).max()
        max_level_name = [k for k, v in alert_rank.items() if v == max_level][0]
    else:
        max_level_name = None

    validation_results.append({
        "CITY": city, "DISEASE": disease, "in_test_period": True,
        "lstm_excluded": disease in LSTM_EXCLUDED_DISEASES,
        "first_alert_date": first_alert_date, "lead_time_weeks": lead_time_weeks,
        "max_alert_level": max_level_name,
        "detected": max_level_name is not None and max_level_name != "Normal",
    })

validation_df = pd.DataFrame(validation_results)
print(validation_df.to_string())

validation_df.to_csv("Task4/task4_validation.csv", index=False)
print("\nSaved: Task4/task4_validation.csv")

             CITY              DISEASE  in_test_period  lstm_excluded first_alert_date  lead_time_weeks max_alert_level  detected
0        Kinshasa  Ebola virus disease            True          False             None              0.0            None     False
1           Lagos          Lassa fever            True          False             None              0.0            None     False
2         Nairobi              Cholera            True          False             None              0.0            None     False
3     Addis Ababa              Malaria            True          False             None              0.0            None     False
4         Kampala  Ebola virus disease            True          False             None              0.0            None     False
5        Khartoum              Cholera            True          False             None              0.0            None     False
6           Cairo             COVID-19            True          False             None    

### 5.3 Recompute Signals per Scenario with Localized DBSCAN Windows

In [38]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 1 (faster version): filter using merge instead of row-wise apply
# ------------------------------------------------------------
scenario_pairs_df = scenarios_df[["CITY", "DISEASE"]].drop_duplicates()

needed_cols = ["CITY", "DISEASE", "VISIT_DATE", "VISIT_YEAR", "VISIT_WEEK",
               "IS_OUTBREAK", "AGE_AT_VISIT_NORM", "COMORBIDITY_COUNT_NORM"] + SYM_COLS

train_full = pd.read_csv("Model 1/artifacts/train_data.csv", usecols=needed_cols)
val_full   = pd.read_csv("Model 1/artifacts/val_data.csv",   usecols=needed_cols)
test_full  = test_df[needed_cols].copy()

all_visits = pd.concat([train_full, val_full, test_full], ignore_index=True)
all_visits["VISIT_DATE"] = pd.to_datetime(all_visits["VISIT_DATE"])

# Fast filter: inner merge keeps only rows matching one of the 12 (city, disease) pairs
scenario_visits = all_visits.merge(scenario_pairs_df, on=["CITY", "DISEASE"], how="inner")
scenario_visits = scenario_visits.sort_values(["CITY", "DISEASE", "VISIT_DATE"]).reset_index(drop=True)

print("Scenario visits shape (full history for these 12 series):", scenario_visits.shape)
print(scenario_visits.groupby(["CITY", "DISEASE"]).size())

Scenario visits shape (full history for these 12 series): (262774, 36)
CITY           DISEASE            
Addis Ababa    Malaria                52876
Cairo          COVID-19               20373
Dar es Salaam  Dengue fever           11279
Goma           Mpox                    4334
Kampala        Ebola virus disease     6235
Kano           Meningitis             24427
Khartoum       Cholera                28578
Kinshasa       Ebola virus disease    12248
Lagos          Lassa fever            24192
Lusaka         Cholera                21394
Mansoura       Typhoid fever          11377
Nairobi        Cholera                45461
dtype: int64


In [39]:
# ------------------------------------------------------------
# Step 2: Recompute DAY_INDEX_NORM (same t0, day_max as before) and
# RECENT_CASE_COUNT (causal, same logic) for this scenario subset
# ------------------------------------------------------------

# DAY_INDEX_NORM — reuse the SAME t0 and day_max computed earlier from train_data.csv
scenario_visits["DAY_INDEX"] = (scenario_visits["VISIT_DATE"] - t0).dt.days.astype(float)
scenario_visits["DAY_INDEX_NORM"] = scenario_visits["DAY_INDEX"] / day_max

# RECENT_CASE_COUNT — same causal two-pointer logic, applied within this subset only
# (safe here because we already filtered to full history per series, no truncation)
CASE_RATE_WINDOW_DAYS = 7
counts = np.zeros(len(scenario_visits))
window_np = np.timedelta64(CASE_RATE_WINDOW_DAYS, "D")

for (city, disease), grp in scenario_visits.groupby(["CITY", "DISEASE"], sort=False):
    dates = grp["VISIT_DATE"].values
    idx   = grp.index.values
    left  = 0
    for right in range(len(dates)):
        while dates[right] - dates[left] > window_np:
            left += 1
        counts[idx[right]] = right - left

scenario_visits["RECENT_CASE_COUNT"] = counts
scenario_visits["RECENT_CASE_COUNT_NORM"] = case_rate_scaler.transform(scenario_visits[["RECENT_CASE_COUNT"]])

print("DAY_INDEX_NORM range:", scenario_visits["DAY_INDEX_NORM"].min(), "->", scenario_visits["DAY_INDEX_NORM"].max())
print("\nRECENT_CASE_COUNT stats:")
print(scenario_visits["RECENT_CASE_COUNT"].describe())

print("\nMean RECENT_CASE_COUNT for outbreak vs background rows (sanity check):")
print(scenario_visits.groupby("IS_OUTBREAK")["RECENT_CASE_COUNT"].mean())

DAY_INDEX_NORM range: 0.0 -> 1.168054047532875

RECENT_CASE_COUNT stats:
count    262774.000000
mean        123.495232
std         194.068295
min           0.000000
25%          18.000000
50%          55.000000
75%         111.000000
max        1548.000000
Name: RECENT_CASE_COUNT, dtype: float64

Mean RECENT_CASE_COUNT for outbreak vs background rows (sanity check):
IS_OUTBREAK
0     58.781782
1    392.547155
Name: RECENT_CASE_COUNT, dtype: float64


In [40]:
# ------------------------------------------------------------
# Step 3: Run Isolation Forest on the scenario subset
# ------------------------------------------------------------
X_iso_scenario = scenario_visits[ISO_FEATURES].values
raw_pred = iso_forest.predict(X_iso_scenario)
scenario_visits["if_pred"] = (raw_pred == -1).astype(int)
scenario_visits["if_score"] = iso_forest.decision_function(X_iso_scenario)

print("IF flagged rate:", scenario_visits["if_pred"].mean())

# ------------------------------------------------------------
# Step 4: Run PCA + DBSCAN on the scenario subset
# ------------------------------------------------------------
pca_cols_scenario = [f"pca_{i}" for i in range(8)]
scenario_visits[pca_cols_scenario] = pca.transform(scenario_visits[SYM_COLS].values)

dbscan_scores_scenario = run_chunked_dbscan(
    scenario_visits, DBSCAN_FEATURES_PCA,
    eps=dbscan_params["eps"], min_samples=dbscan_params["min_samples"]
)
scenario_visits["dbscan_score"] = dbscan_scores_scenario
scenario_visits["dbscan_pred"] = (scenario_visits["dbscan_score"] >= DBSCAN_THRESHOLD).astype(int)

print("DBSCAN flagged rate:", scenario_visits["dbscan_pred"].mean())

print("\nBoth models done. Sample:")
print(scenario_visits[["CITY", "DISEASE", "VISIT_DATE", "IS_OUTBREAK", "if_pred", "dbscan_pred"]].head())

IF flagged rate: 0.0369442943365782
DBSCAN flagged rate: 0.0

Both models done. Sample:
          CITY  DISEASE          VISIT_DATE  IS_OUTBREAK  if_pred  dbscan_pred
0  Addis Ababa  Malaria 2000-01-01 07:43:24            0        0            0
1  Addis Ababa  Malaria 2000-01-01 14:19:56            0        0            0
2  Addis Ababa  Malaria 2000-01-03 02:29:44            0        0            0
3  Addis Ababa  Malaria 2000-01-04 22:35:51            0        0            0
4  Addis Ababa  Malaria 2000-01-05 18:58:21            0        0            0


In [41]:
# ------------------------------------------------------------
# Fix: restrict each scenario's chunk to a window around its outbreak period
# before running DBSCAN (keeps DAY_INDEX_NORM density comparable to the
# original test-period chunks). RECENT_CASE_COUNT stays as already computed
# (needs full history for correctness), only the DBSCAN clustering window shrinks.
# ------------------------------------------------------------

windowed_chunks = []
for _, row in scenarios_df.iterrows():
    city, disease = row["CITY"], row["DISEASE"]
    window_start = row["exact_start"] - pd.Timedelta(days=90)
    window_end   = row["exact_end"] + pd.Timedelta(days=30)

    chunk = scenario_visits[
        (scenario_visits["CITY"] == city) & (scenario_visits["DISEASE"] == disease) &
        (scenario_visits["VISIT_DATE"] >= window_start) & (scenario_visits["VISIT_DATE"] <= window_end)
    ].copy()
    windowed_chunks.append(chunk)

scenario_visits_windowed = pd.concat(windowed_chunks, ignore_index=True)
print("Windowed scenario visits shape:", scenario_visits_windowed.shape)
print(scenario_visits_windowed.groupby(["CITY", "DISEASE"]).size())

print("\nDAY_INDEX_NORM range within windows (should be narrow now):")
print(scenario_visits_windowed.groupby(["CITY","DISEASE"])["DAY_INDEX_NORM"].agg(lambda x: x.max()-x.min()))

# Re-run DBSCAN on the windowed subset only
dbscan_scores_windowed = run_chunked_dbscan(
    scenario_visits_windowed, DBSCAN_FEATURES_PCA,
    eps=dbscan_params["eps"], min_samples=dbscan_params["min_samples"]
)
scenario_visits_windowed["dbscan_score"] = dbscan_scores_windowed
scenario_visits_windowed["dbscan_pred"] = (scenario_visits_windowed["dbscan_score"] >= DBSCAN_THRESHOLD).astype(int)

print("\nDBSCAN flagged rate (windowed):", scenario_visits_windowed["dbscan_pred"].mean())

Windowed scenario visits shape: (48760, 52)
CITY           DISEASE            
Addis Ababa    Malaria                7233
Cairo          COVID-19               7438
Dar es Salaam  Dengue fever           3412
Goma           Mpox                   2599
Kampala        Ebola virus disease    2183
Kano           Meningitis             2260
Khartoum       Cholera                4705
Kinshasa       Ebola virus disease    3390
Lagos          Lassa fever            3438
Lusaka         Cholera                3734
Mansoura       Typhoid fever          1921
Nairobi        Cholera                6447
dtype: int64

DAY_INDEX_NORM range within windows (should be narrow now):
CITY           DISEASE            
Addis Ababa    Malaria                0.025093
Cairo          COVID-19               0.028592
Dar es Salaam  Dengue fever           0.028471
Goma           Mpox                   0.028230
Kampala        Ebola virus disease    0.024611
Kano           Meningitis             0.024852
Khartoum      

In [43]:
# ------------------------------------------------------------
# Weekly aggregation for the scenario windows: IF (binary flag) + DBSCAN (descriptive score)
# ------------------------------------------------------------
scenario_visits_windowed["WEEK_START"] = (
    scenario_visits_windowed["VISIT_DATE"] - pd.to_timedelta(scenario_visits_windowed["VISIT_DATE"].dt.weekday, unit="D")
).dt.normalize()

scenario_weekly = scenario_visits_windowed.groupby(["CITY", "DISEASE", "WEEK_START"]).agg(
    if_flagged_visits=("if_pred", "sum"),
    dbscan_score_mean=("dbscan_score", "mean"),
    total_visits=("if_pred", "count"),
    true_outbreak=("IS_OUTBREAK", "max"),
).reset_index()

scenario_weekly["IF_FLAG"] = (scenario_weekly["if_flagged_visits"] >= 1).astype(int)

print("scenario_weekly shape:", scenario_weekly.shape)
print(scenario_weekly.head())

scenario_weekly shape: (378, 8)
          CITY  DISEASE WEEK_START  if_flagged_visits  dbscan_score_mean  \
0  Addis Ababa  Malaria 2021-05-31                  1         530.176072   
1  Addis Ababa  Malaria 2021-06-07                  2         530.176072   
2  Addis Ababa  Malaria 2021-06-14                  2         519.572550   
3  Addis Ababa  Malaria 2021-06-21                  2         530.176072   
4  Addis Ababa  Malaria 2021-06-28                  0         530.176072   

   total_visits  true_outbreak  IF_FLAG  
0            31              0        1  
1            55              0        1  
2            50              0        1  
3            51              0        1  
4            42              0        0  


### 5.4 Recompute LSTM Predictions for the Scenario Series

In [44]:
# ------------------------------------------------------------
# Reuse the FULL windowed arrays (train+val+test combined) already built earlier
# for the weekly LSTM data — just filter to the 12 scenario series
# ------------------------------------------------------------
scenario_city_disease = set(zip(scenarios_df["CITY"], scenarios_df["DISEASE"]))

def group_matches_scenario(group_str):
    parts = group_str.split("|")
    if len(parts) != 3:
        return False
    _, city, disease = parts
    return (city, disease) in scenario_city_disease

scenario_mask = np.array([group_matches_scenario(g) for g in group_all])
print("Rows matching scenario series (any split, any date):", scenario_mask.sum())

if scenario_mask.sum() > 0:
    scenario_lstm_input = {
        "seq_numeric": X_seq_numeric[scenario_mask], "season_seq": X_season[scenario_mask],
        "country": X_country[scenario_mask], "city": X_city[scenario_mask],
        "disease": X_disease[scenario_mask], "latlon": X_latlon[scenario_mask],
    }
    probs_per_model = [m.predict(scenario_lstm_input, verbose=0).ravel() for m in ensemble_models]
    scenario_lstm_prob = np.mean(probs_per_model, axis=0)
    scenario_lstm_pred = (scenario_lstm_prob >= LSTM_THRESHOLD).astype(int)

    scenario_lstm_df = pd.DataFrame({
        "group": group_all[scenario_mask],
        "target_date": target_dates[scenario_mask],
        "lstm_prob": scenario_lstm_prob,
        "lstm_pred": scenario_lstm_pred,
    })
    scenario_lstm_df[["COUNTRY", "CITY", "DISEASE"]] = scenario_lstm_df["group"].str.split("|", expand=True)
    print("scenario_lstm_df shape:", scenario_lstm_df.shape)
    print(scenario_lstm_df.groupby(["CITY","DISEASE"]).size())
else:
    scenario_lstm_df = pd.DataFrame(columns=["target_date","CITY","DISEASE","lstm_prob","lstm_pred"])
    print("No LSTM rows found (all 12 might be excluded or unmatched)")

Rows matching scenario series (any split, any date): 12760
scenario_lstm_df shape: (12760, 7)
CITY           DISEASE            
Addis Ababa    Malaria                1376
Cairo          COVID-19                384
Dar es Salaam  Dengue fever           1374
Kampala        Ebola virus disease    1375
Kano           Meningitis             1374
Khartoum       Cholera                1376
Kinshasa       Ebola virus disease    1374
Lagos          Lassa fever            1376
Lusaka         Cholera                1376
Nairobi        Cholera                1375
dtype: int64


### 5.5 Final Fusion & Validation Result

In [45]:
# ------------------------------------------------------------
# Final merge: IF (binary) + DBSCAN (descriptive score) + LSTM (binary, where available)
# ------------------------------------------------------------
scenario_lstm_df["target_date"] = pd.to_datetime(scenario_lstm_df["target_date"])

final_scenario_alerts = pd.merge(
    scenario_weekly,
    scenario_lstm_df[["CITY", "DISEASE", "target_date", "lstm_prob", "lstm_pred"]],
    left_on=["CITY", "DISEASE", "WEEK_START"], right_on=["CITY", "DISEASE", "target_date"],
    how="left"
)
final_scenario_alerts["lstm_pred"] = final_scenario_alerts["lstm_pred"].fillna(-1).astype(int)  # -1 = N/A (excluded disease)

# Fusion score: only count LSTM if it's actually available (not -1)
final_scenario_alerts["fusion_score"] = (
    final_scenario_alerts["IF_FLAG"] * 1 +
    final_scenario_alerts["lstm_pred"].clip(lower=0) * 2
)
# DBSCAN treated as descriptive (not part of the binary score here, per our agreed approach)

def assign_alert_scenario(row):
    score = row["fusion_score"]
    if score == 0: return "Normal"
    elif score <= 2: return "Watch"
    elif score == 3: return "Warning"
    else: return "Emergency"

final_scenario_alerts["alert_level"] = final_scenario_alerts.apply(assign_alert_scenario, axis=1)
final_scenario_alerts["lstm_available"] = final_scenario_alerts["lstm_pred"] != -1

# ------------------------------------------------------------
# Validate against each scenario's exact dates
# ------------------------------------------------------------
alert_rank = {"Normal": 0, "Watch": 1, "Warning": 2, "Emergency": 3}
final_results = []

for _, row in scenarios_df.iterrows():
    city, disease = row["CITY"], row["DISEASE"]
    exact_start, exact_end = row["exact_start"], row["exact_end"]

    series_alerts = final_scenario_alerts[
        (final_scenario_alerts["CITY"] == city) & (final_scenario_alerts["DISEASE"] == disease)
    ].sort_values("WEEK_START")

    lookback = series_alerts[
        (series_alerts["WEEK_START"] >= exact_start - pd.Timedelta(weeks=8)) &
        (series_alerts["WEEK_START"] < exact_start)
    ]
    warned = lookback[lookback["alert_level"] != "Normal"]
    if len(warned) > 0:
        first_alert = warned["WEEK_START"].min()
        lead_weeks = (exact_start - first_alert).days / 7
    else:
        first_alert, lead_weeks = None, 0

    during = series_alerts[(series_alerts["WEEK_START"] >= exact_start) & (series_alerts["WEEK_START"] <= exact_end)]
    if len(during) > 0:
        max_lvl = during["alert_level"].map(alert_rank).max()
        max_lvl_name = [k for k,v in alert_rank.items() if v == max_lvl][0]
        detection_rate = (during["alert_level"] != "Normal").mean()
    else:
        max_lvl_name, detection_rate = None, None

    final_results.append({
        "CITY": city, "DISEASE": disease,
        "lstm_available": disease not in LSTM_EXCLUDED_DISEASES,
        "first_alert_date": first_alert, "lead_time_weeks": lead_weeks,
        "max_alert_level": max_lvl_name, "detection_rate": detection_rate,
        "detected": max_lvl_name is not None and max_lvl_name != "Normal",
    })

final_validation_df = pd.DataFrame(final_results)
print(final_validation_df.to_string())

final_validation_df.to_csv("Task4/task4_validation_FINAL.csv", index=False)
print("\nSaved: Task4/task4_validation_FINAL.csv")

             CITY              DISEASE  lstm_available first_alert_date  lead_time_weeks max_alert_level  detection_rate  detected
0        Kinshasa  Ebola virus disease            True       2020-06-08         7.857143         Warning        1.000000      True
1           Lagos          Lassa fever            True       2020-11-16         6.714286         Warning        1.000000      True
2         Nairobi              Cholera            True       2021-02-22         5.428571         Warning        1.000000      True
3     Addis Ababa              Malaria            True       2021-07-19         6.285714         Warning        1.000000      True
4         Kampala  Ebola virus disease            True       2022-08-08         3.714286         Warning        1.000000      True
5        Khartoum              Cholera            True       2022-05-23         5.571429         Warning        1.000000      True
6           Cairo             COVID-19            True       2020-11-16         6.7

## 6. Final Task 4 Summary

In [46]:
import json

# ------------------------------------------------------------
# Build the final Task 4 summary (task4_summary.json)
# ------------------------------------------------------------
task4_summary = {
    "scenarios_detected": "12/12 (100%)",
    "average_lead_time_weeks": 5.04,
    "average_detection_rate_during_outbreak": 0.972,
    "alert_level_counts_at_peak": {"Warning": 10, "Watch": 2, "Emergency": 0, "Normal": 0},

    "fusion_system": {
        "weights": {"Isolation_Forest": 1, "DBSCAN": 2, "LSTM": 2},
        "thresholds": {"Watch": "1-2 points", "Warning": "3 points", "Emergency": "4-5 points"},
        "note": "DBSCAN excluded from the binary scenario-validation score due to a "
                "threshold-portability limitation discovered when re-clustering narrow "
                "time windows (see report limitations section). Used as descriptive "
                "signal only in that specific validation."
    },

    "known_limitations": [
        "9 low-signal diseases (incl. Mpox, Typhoid fever) excluded from LSTM training "
        "due to insufficient positive examples -> those 2 of 12 scenarios rely on IF only, "
        "capping them at Watch level.",
        "DBSCAN's absolute score threshold (635.89) was calibrated on the full test-period "
        "clustering context and is not directly portable to narrower scenario-specific "
        "clustering windows -- a methodological property of density-based clustering, "
        "not a data quality issue.",
        "Kano/Meningitis scenario had a very short lead time (0.29 weeks) -- flagged for "
        "discussion in the report.",
        "LSTM's own held-out test metrics: F1=0.644, PR-AUC=0.567 (single-week-ahead target).",
        "DBSCAN's own held-out test metrics: ROC-AUC=0.921, FPR=0.014, Precision=0.489, Recall=0.460.",
        "Isolation Forest's own held-out test metrics: F1=0.123, but delivers the longest "
        "genuine early-warning lead time (~35 days average) of the three models.",
    ],

    "overall_weekly_alert_precision": {
        "Watch": 0.0025,
        "Warning": 0.3871,
        "Emergency": 0.90
    }
}

with open("Task4/task4_summary.json", "w") as f:
    json.dump(task4_summary, f, indent=2, default=str)

print(json.dumps(task4_summary, indent=2, default=str))
print("\nSaved: Task4/task4_summary.json")

{
  "scenarios_detected": "12/12 (100%)",
  "average_lead_time_weeks": 5.04,
  "average_detection_rate_during_outbreak": 0.972,
  "alert_level_counts_at_peak": {
    "Warning": 10,
    "Watch": 2,
    "Emergency": 0,
    "Normal": 0
  },
  "fusion_system": {
    "weights": {
      "Isolation_Forest": 1,
      "DBSCAN": 2,
      "LSTM": 2
    },
    "thresholds": {
      "Watch": "1-2 points",
      "Warning": "3 points",
      "Emergency": "4-5 points"
    },
    "note": "DBSCAN excluded from the binary scenario-validation score due to a threshold-portability limitation discovered when re-clustering narrow time windows (see report limitations section). Used as descriptive signal only in that specific validation."
  },
  "known_limitations": [
    "9 low-signal diseases (incl. Mpox, Typhoid fever) excluded from LSTM training due to insufficient positive examples -> those 2 of 12 scenarios rely on IF only, capping them at Watch level.",
    "DBSCAN's absolute score threshold (635.89) was